# Анализ конверсии сессий

Классификация завершённых сессий Online Shoppers Purchasing Intention в Cart Scope.

Создать пропуски: `python -m src.data.create_missing_sessions`.

## Содержание

1. [Данные и пропуски](#data)
2. [Разделение сессий](#split)
3. [Признак числа пропусков](#missing)
4. [Конвейер](#pipeline)
5. [Подбор параметров](#search)
6. [Оценка на отложенных данных](#evaluation)
7. [Сохранение результатов](#results)
8. [Ограничения](#limitations)

In [ ]:
import hashlib
import warnings
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split


import matplotlib.pyplot as plt


from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display, Markdown

warnings.filterwarnings("error", category=ConvergenceWarning)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data/raw/online_shoppers_intention.csv"
DATA = ROOT / "data/processed/online_shoppers_with_missing.csv"
REPORTS = ROOT / "reports/conversion"
MODEL = ROOT / "models/session_conversion.pkl"

<a id="data"></a>

## Данные и пропуски

В исходных данных 17 признаков и целевой флаг `Revenue`.
Числовые коды ОС, браузера, региона и источника трафика являются категориями.

In [ ]:
NUMERIC = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
CATEGORICAL = [
    "Month", "OperatingSystems", "Browser", "Region", "TrafficType",
    "VisitorType", "Weekend",
]


raw = pd.read_csv(RAW)
data = pd.read_csv(DATA)
display(data.head())
display(pd.DataFrame({"тип": data.dtypes.astype(str), "пропуски": data.isna().sum(), "уникальных": data.nunique()}))
print("Исходных строк:", len(raw), "Всего строк:", len(data))
print("Полных повторов в исходном CSV:", raw.duplicated().sum())
display(data[NUMERIC].describe().T)
display(raw["Revenue"].value_counts(normalize=True).rename("доля"))

<a id="split"></a>

## Разделение сессий

Разделяем данные на обучение и тест в соотношении 80/20 со стратификацией по `Revenue`.

In [ ]:
X = data.drop(columns="Revenue")
y = data.Revenue.astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
display(pd.DataFrame({
    "строк": [len(X_train), len(X_test)],
    "доля покупок": [y_train.mean(), y_test.mean()],
}, index=["train", "test"]))

<a id="missing"></a>

## Признак числа пропусков

Трансформер вычисляет `missing_count` по пустым значениям до их заполнения.
Этот признак создаётся внутри конвейера и не записывается в исходный CSV.

In [ ]:
class MissingCountTransformer(TransformerMixin, BaseEstimator):
    """Сохранить исходные столбцы и добавить missing_count."""

    def fit(self, X, y=None):
        if "missing_count" in X.columns:
            raise ValueError("Столбец missing_count уже существует.")
        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = len(X.columns)
        return self

    def transform(self, X):
        check_is_fitted(self)
        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError("Состав или порядок признаков отличается от обучения.")
        result = X.copy()
        result["missing_count"] = X.isna().sum(axis=1)
        return result


In [ ]:
display(MissingCountTransformer().fit_transform(data.drop(columns="Revenue")).tail())


<a id="pipeline"></a>

## Конвейер

Числовые пропуски заполняются медианой, затем применяется StandardScaler.
Категориальные пропуски заполняются модой, затем применяется OneHotEncoder.
Неизвестные категории допускаются. Все обучаемые преобразования находятся внутри Pipeline.

In [ ]:
numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
pipeline = Pipeline([
    ("missing_count", MissingCountTransformer()),
    ("preprocessor", ColumnTransformer([
        ("num", numeric, NUMERIC + ["missing_count"]),
        ("cat", categorical, CATEGORICAL),
    ])),
    ("classifier", LogisticRegression(max_iter=3000, random_state=42)),
])

pipeline


<a id="search"></a>

## Подбор параметров

Сравниваем четыре значения `C` и два варианта `class_weight` на пяти
стратифицированных фолдах. Критерий выбора – Average Precision из-за дисбаланса классов.
Медиана остаётся фиксированной. Тестовая выборка в подборе не участвует.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline,
    {"classifier__C": [0.01, 0.1, 1.0, 10.0],
     "classifier__class_weight": [None, "balanced"]},
    scoring={"average_precision": "average_precision", "roc_auc": "roc_auc", "f1": "f1"},
    refit="average_precision", cv=cv, n_jobs=1, error_score="raise",
)
search.fit(X_train, y_train)
print("best_params_ =", search.best_params_)
print("Средняя Average Precision на фолдах:", search.best_score_)
results = pd.DataFrame(search.cv_results_)
columns = [c for c in results if c.startswith(("param_", "mean_test_", "std_test_", "rank_test_"))]
display(results[columns].sort_values("rank_test_average_precision"))

<a id="evaluation"></a>

## Оценка на отложенных данных

Сравниваем выбранную модель с постоянным прогнозом «нет покупки». Порог – 0,5.

In [ ]:
def evaluate(model, X, y):
    prediction = model.predict(X)
    probability = model.predict_proba(X)[:, 1]
    return {
        "rows": len(y), "positives": int(y.sum()),
        "accuracy": accuracy_score(y, prediction),
        "precision": precision_score(y, prediction, zero_division=0),
        "recall": recall_score(y, prediction, zero_division=0),
        "f1": f1_score(y, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y, probability),
        "average_precision": average_precision_score(y, probability),
        "confusion_matrix": confusion_matrix(y, prediction, labels=[0, 1]).tolist(),
    }


In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
metrics = {
    "seed": 42, "best_params": search.best_params_,
    "best_cv_average_precision": search.best_score_,
    "train_rows": len(X_train),
    "test": evaluate(search.best_estimator_, X_test, y_test),
    "baseline": evaluate(baseline, X_test, y_test),
    "input_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (RAW, DATA)},
}
display(pd.DataFrame({name: {k: v for k, v in metrics[name].items() if k != "confusion_matrix"}
                      for name in ["test", "baseline"]}).T)

<a id="results"></a>

## Сохранение результатов

Сохраняем полный конвейер через cloudpickle: он включает класс трансформера,
определённый в ноутбуке. Для загрузки используйте `cloudpickle.load` в том же окружении.
Повторный запуск обновляет модель, таблицы и график.

In [ ]:
REPORTS.mkdir(parents=True, exist_ok=True)
MODEL.parent.mkdir(parents=True, exist_ok=True)
with MODEL.open("wb") as stream:
    cloudpickle.dump(search.best_estimator_, stream)
results = pd.DataFrame(search.cv_results_)
columns = [c for c in results if c.startswith(("param_", "mean_test_", "std_test_", "rank_test_"))]
ConfusionMatrixDisplay.from_predictions(y_test, search.predict(X_test), display_labels=["Без покупки", "Покупка"], cmap="Blues")
plt.title("Конверсия: отложенные сессии")
plt.xlabel("Прогноз")
plt.ylabel("Фактический класс")
plt.tight_layout()
plt.savefig(REPORTS / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
lines = ["# Анализ конверсии сессий", "", f"Лучшие параметры: `{search.best_params_}`.",
         f"Средняя Average Precision на 5 фолдах: {search.best_score_:.4f}.", "",
         "| Выборка | Строк | Precision | Recall | F1 | ROC-AUC | Average Precision |",
         "| --- | ---: | ---: | ---: | ---: | ---: | ---: |"]
for name, key in [("Тест", "test"), ("Всегда без покупки", "baseline")]:
    m = metrics[key]
    lines.append(f"| {name} | {m['rows']} | {m['precision']:.4f} | {m['recall']:.4f} | {m['f1']:.4f} | {m['roc_auc']:.4f} | {m['average_precision']:.4f} |")
lines += ["", f"Обучение: {len(X_train)} строк. Разделение 80/20 со стратификацией, seed=42.",
          "", "Подбор C и class_weight: 8 комбинаций, 5 фолдов. Критерий – Average Precision. "
          "Предобработка обучается внутри Pipeline. Порог классификации – 0,5.",
          "", "Оценка может быть завышена: похожие искусственные копии и дубликаты "
          "при случайном разделении могут попасть в разные выборки и фолды. "
          "Результаты относятся к завершённым сессиям, а не к прогнозу до покупки.", ""]
(REPORTS / "summary.md").write_text("\n".join(lines), encoding="utf-8")
display(Markdown("\n".join(lines)))

<a id="limitations"></a>

## Ограничения

Результаты относятся к завершённым сессиям. Доступность `PageValues` и итоговых
агрегатов до покупки не подтверждена, поэтому это не оценка прогноза в реальном времени.
Искусственные пропуски не обязательно отражают реальные причины отсутствия данных.
Случайное разделение не измеряет качество на будущих месяцах.

Дубликаты и похожие копии при случайном разделении могут попасть в разные выборки
и фолды, поэтому оценка может быть завышена.